

<div align="center">
  <h1>House Price Prediction</h1>
</div>

In [21]:
import pandas as p
p.set_option('display.max_columns', 100)
p.set_option('display.max_rows', 100)

In [32]:
htrain = train_df
htest = test_df

In [34]:
htrain.shape, htest.shape
# ((16512, 10), (4128, 10))
htrain.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
17875,-121.99,37.40,35.0,1845.0,325.0,1343.0,317.0,5.3912,235300.0,<1H OCEAN
9360,-122.53,37.95,22.0,7446.0,1979.0,2980.0,1888.0,3.5838,271300.0,NEAR BAY
4338,-118.31,34.08,26.0,1609.0,534.0,1868.0,497.0,2.7038,227100.0,<1H OCEAN
986,-121.85,37.72,43.0,228.0,40.0,83.0,42.0,10.3203,400000.0,INLAND
8129,-118.17,33.80,26.0,1589.0,380.0,883.0,366.0,3.5313,187500.0,NEAR OCEAN


In [37]:
null_value_cols = [col for col in htrain.columns if htrain[col].isnull().any()]

In [39]:
htrain[null_value_cols].isnull().sum()

total_bedrooms    169
dtype: int64

In [42]:
htrain = htrain.drop(columns=['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'], errors='ignore')

In [44]:
htest = htest.drop(columns=['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'], errors='ignore')

In [46]:
htrain[null_value_cols].isnull().sum()

total_bedrooms    169
dtype: int64

In [50]:
htrain[null_value_cols].dtypes
htrain = htrain.drop(columns='LotFrontage', errors='ignore')

In [52]:
fresh_housing_train = htrain.dropna()
fresh_housing_train.shape

(16343, 10)

In [53]:
low_cardinality_col = [col for col in fresh_housing_train.columns if
fresh_housing_train[col].dtype == object and
fresh_housing_train[col].nunique() < 10]
high_cardinality_col = [col for col in fresh_housing_train.columns if
fresh_housing_train[col].dtype == object and
fresh_housing_train[col].nunique() >= 10]
num_col = [col for col in fresh_housing_train.columns if
fresh_housing_train[col].dtype in [int, float]]

In [55]:
new_housing_train = p.concat([fresh_housing_train[low_cardinality_col],
fresh_housing_train[high_cardinality_col],
fresh_housing_train[num_col]], axis=1)

In [58]:
new_housing_train = new_housing_train.drop(columns='Id', errors='ignore')
htest = htest.drop(columns=['LotFrontage', 'Id'], errors='ignore')

In [61]:
null_value_cols = [col for col in htest.columns if htest[col].isnull().any()]
htest[htest.isnull().any(axis=1)][null_value_cols]  # Prints all the null value columns

fresh_housing_test = htest.dropna()
low_cardinality_col = [col for col in fresh_housing_test.columns if
	fresh_housing_test[col].dtype == object and
	fresh_housing_test[col].nunique() < 10]
high_cardinality_col = [col for col in fresh_housing_test.columns if
	fresh_housing_test[col].dtype == object and
	fresh_housing_test[col].nunique() >= 10]
num_col = [col for col in fresh_housing_test.columns if
	fresh_housing_test[col].dtype in [int, float]]
new_housing_test = p.concat([fresh_housing_test[low_cardinality_col],
							 fresh_housing_test[high_cardinality_col],
							 fresh_housing_test[num_col]], axis=1)

In [62]:
copy_of_housing_train = new_housing_train.copy()
copy_of_housing_test = new_housing_test.copy()

In [64]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_col = low_cardinality_col + high_cardinality_col
for col in cat_col:
	new_housing_train[col] = le.fit_transform(new_housing_train[col])
for col in cat_col:
	new_housing_test[col] = le.fit_transform(new_housing_test[col])

In [66]:
from sklearn.model_selection import train_test_split
y = new_housing_train['median_house_value']
X = new_housing_train.drop(columns='median_house_value')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=766)
X_train.shape, X_test.shape

((13074, 9), (3269, 9))

In [67]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
dtr = DecisionTreeRegressor()
rfr = RandomForestRegressor()
dtr.fit(X_train, y_train)
dtr_prediction = dtr.predict(X_test)
rfr.fit(X_train, y_train)
rfr_prediction = rfr.predict(X_test)

In [69]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
dtr_error = mean_absolute_error(y_test, dtr_prediction)
dtr_error_percentage = mean_absolute_percentage_error(y_test,
dtr_prediction)
dtr_error, dtr_error_percentage*100 #(27336.078358208953, 14.489400761489787)

(45313.9323952279, 24.347791406410593)

In [70]:
rfr_error = mean_absolute_error(y_test, rfr_prediction)
rfr_error_percentage = mean_absolute_percentage_error(y_test,rfr_prediction)
rfr_error, rfr_error_percentage*100 # (18731.851417910446, 10.288181221800267)

(32545.8322789844, 18.109059804056717)